In [0]:
from pyspark.sql.types import *

people_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", StringType(), True),
    StructField("salary", StringType(), True),
    StructField("address", StringType(), True),
    StructField("gender", StringType(), True)
])



In [0]:
df = spark.read.format("csv")\
    .option("header","false")\
    .option("inferSchema","true")\
    .schema(people_schema)\
    .option("skipRows",1)\
    .option("mode","Permissive")\
    .load("/Volumes/spark_data/dev/spark_data_volume/people_data.csv")

df.show()

In [0]:
df.write.format("csv")\
    .option("header",'true')\
    .mode('overwrite')\
    .option("path","/Volumes/spark_data/dev/spark_data_volume/people_data_new/")\
    .save()

In [0]:
df.repartition(3).write.format("csv")\
    .option("header",'true')\
    .mode('overwrite')\
    .option("path","/Volumes/spark_data/dev/spark_data_volume/people_data_new/")\
    .save()

In [0]:
df.write.format("csv")\
    .option("header",'true')\
    .mode('overwrite')\
    .partitionBy("address")\
    .option("path","/Volumes/spark_data/dev/spark_data_volume/people_data/partition_by_address/")\
    .save()

In [0]:
df.write.format("csv")\
    .option("header",'true')\
    .mode('overwrite')\
    .partitionBy("address","gender")\
    .option("path","/Volumes/spark_data/dev/spark_data_volume/people_data/partition_by_address_gender/")\
    .save()

In [0]:
display(dbutils.fs.ls("/Volumes/spark_data/dev/spark_data_volume/people_data/partition_by_address_gender/"))

In [0]:
display(dbutils.fs.ls("/Volumes/spark_data/dev/spark_data_volume/people_data/partition_by_address"))

In [0]:
display(dbutils.fs.ls("/Volumes/spark_data/dev/spark_data_volume/people_data_new.csv"))

In [0]:
display(dbutils.fs.ls("/Volumes/spark_data/dev/spark_data_volume/people_data_new"))

## Bucket By

In [0]:
df.write.format("csv")\
    .option("header",'true')\
    .mode('overwrite')\
    .bucketBy(3,"id")\
    .saveAsTable('spark_data.dev.people_data_bucketed')

In [0]:
data =[
(1,1),
(2,1),
(3,1),
(4,2),
(5,1),
(6,2),
(7,2)
]

In [0]:
schema = ['id','num']

In [0]:
spark.createDataFrame(data = data,schema= schema).show()

In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import *
df.select("name","address").show()
df.select(col("name"),col("address")).show()

In [0]:
df.select("name",col("age"),df["salary"],df.id).show()

# expression

In [0]:
df.select(expr("id + 5")).show()

In [0]:
df.select(expr("id as employee_id"),expr("name as employee_name"),
          expr("concat(name,address)")).show()

#sparkSQL

In [0]:
df.createOrReplaceTempView("employee")
spark.sql("select * from employee").show()

In [0]:
spark.sql("""
          
          select id,name,age, concat(name, age)
          from employee
          
          """).show()

In [0]:
df.select(col("name").alias("employee_name")).show()

In [0]:
df.filter(col("salary") >= 150000).show()

In [0]:
df.where((col("salary") >= 150000) & (col("age") <= 18)).show()

In [0]:
df.select("*",lit(1000).alias("bonus")).show()
df.select("*",lit(df["salary"]*0.1).alias("newbonus")).show()

adding columns

In [0]:
df.withColumn('sur_name',lit('sahani')).show()

In [0]:
df.withColumnRenamed("id","employee_id").show()

In [0]:
df.withColumn("id",col("id").cast("string")).printSchema()

In [0]:
df.drop("id",col("name")).show()

In [0]:
spark.sql("""
          select *,"kumar" as last_name, concat(name,last_name ) as full_name from employee where salary >= 150000 and age <= 18
          
          """).show()

# Union vs Union ALL

In [0]:
data=[(10 ,'Anil',50000, 18),
(11 ,'Vikas',75000,  16),
(12 ,'Nisha',40000,  18),
(13 ,'Nidhi',60000,  17),
(14 ,'Priya',80000,  18),
(15 ,'Mohit',45000,  18),
(16 ,'Rajesh',90000, 10),
(17 ,'Raman',55000, 16),
(18 ,'Sam',65000,   17)]

manager_df = spark.createDataFrame(data, schema = ["id", "name", "salary", "manger_id"])
manager_df.show()


In [0]:
manager_df.count()

data1=[(19 ,'Sohan',50000, 18),
(20 ,'Sima',75000,  17),
(20,'Sima',75000,  17),
(19,'Sohan',50000,18)]

manager_df2 = spark.createDataFrame(data1, schema = ["id", "Name", "sal","mngr_id"])
manager_df2.show()


In [0]:
manager_df.union(manager_df2).show()

In [0]:
manager_df.union(manager_df2).show()

In [0]:
manager_df.union(manager_df2).count()

In [0]:
manager_df.unionAll(manager_df2).count()

In [0]:
manager_df.createOrReplaceTempView("m1")
manager_df2.createOrReplaceTempView("m2")

In [0]:
spark.sql("""
          select * from m1 union all select * from m2;
          
          """).count()



In [0]:
spark.sql("""
          select * from m1 union select * from m2;
          """).count()

In [0]:
manager_df.unionByName(df).show()